In [2]:
#listing all partitions that exists

partitions = mssparkutils.fs.ls("Files/bronze/f_sls_t")
for p in partitions:
    print(p.name)


StatementMeta(, 4db4e963-533c-4d89-8adf-fde8223bfd86, 4, Finished, Available, Finished, False)

day=20260731
day=20260804
day=20260805
day=20260806
day=20260810
day=20260811


In [3]:
df_fact_sample = spark.read.parquet("Files/bronze/f_sls_t/day=20260804")
df_fact_sample.printSchema()
df_fact_sample.show(10,truncate=False)
print("Row count (Single Partition):", df_fact_sample.count())

StatementMeta(, 4db4e963-533c-4d89-8adf-fde8223bfd86, 5, Finished, Available, Finished, False)

root
 |-- trxn_id: integer (nullable = true)
 |-- prod_num: string (nullable = true)
 |-- store_num: string (nullable = true)
 |-- cust_num: string (nullable = true)
 |-- trxn_dt: timestamp (nullable = true)
 |-- qty: integer (nullable = true)
 |-- amount: decimal(10,2) (nullable = true)

+-------+--------+---------+--------+-------------------+---+------+
|trxn_id|prod_num|store_num|cust_num|trxn_dt            |qty|amount|
+-------+--------+---------+--------+-------------------+---+------+
|548    |P030    |S001     |C054    |2026-08-04 11:43:41|4  |347.68|
|549    |P005    |S001     |C103    |2026-08-04 15:56:49|1  |6.52  |
|550    |P006    |S001     |C060    |2026-08-04 10:18:53|4  |63.04 |
|551    |P017    |S001     |C175    |2026-08-04 07:11:38|1  |91.15 |
|554    |P002    |S001     |C080    |2026-08-04 19:39:47|4  |330.12|
|555    |P016    |S001     |C055    |2026-08-04 13:58:47|4  |295.08|
|557    |P020    |S001     |C134    |2026-08-04 09:18:05|1  |70.79 |
|558    |P002    |S0

In [4]:
from pyspark.sql.functions import col, round as spark_round, current_timestamp, lit

# read all existing partitions at once
df_fact = spark.read.parquet("Files/bronze/f_sls_t")

# no casting needed
# round amount 
df_fact_silver = df_fact.withColumn("amount", spark_round(col("amount"), 2))

# filtering invalid rows before FK validation
df_fact_silver = df_fact_silver.filter(
    col("trxn_id").isNotNull() &
    col("prod_num").isNotNull() &
    col("cust_num").isNotNull() &
    col("store_num").isNotNull()
)


StatementMeta(, 4db4e963-533c-4d89-8adf-fde8223bfd86, 6, Finished, Available, Finished, False)

In [5]:
#FK validation againhts silver dimension // quarantine , here not dropping records

df_dim_cust  = spark.table("silver_customer").select("cust_num").distinct()
df_dim_prod  = spark.table("silver_product").select("prod_num").distinct()
df_dim_store = spark.table("silver_store").select("store_num").distinct()

df_fact_checked = (
    df_fact_silver
    .join(df_dim_cust,  "cust_num",  "left")
    .withColumn("valid_cust", col("cust_num").isin([r["cust_num"] for r in df_dim_cust.collect()]))
)

StatementMeta(, 4db4e963-533c-4d89-8adf-fde8223bfd86, 7, Finished, Available, Finished, False)

In [6]:
df_dim_cust  = spark.table("silver_customer").select("cust_num").withColumnRenamed("cust_num", "cust_num_dim")
df_dim_prod  = spark.table("silver_product").select("prod_num").withColumnRenamed("prod_num", "prod_num_dim")
df_dim_store = spark.table("silver_store").select("store_num").withColumnRenamed("store_num", "store_num_dim")

df_fact_checked = (
    df_fact_silver.alias("f")
    .join(df_dim_cust.alias("dc"),  col("f.cust_num")  == col("dc.cust_num_dim"),  "left")
    .join(df_dim_prod.alias("dp"),  col("f.prod_num")  == col("dp.prod_num_dim"),  "left")
    .join(df_dim_store.alias("ds"), col("f.store_num") == col("ds.store_num_dim"), "left")
    .withColumn("fk_valid",
        col("cust_num_dim").isNotNull() &
        col("prod_num_dim").isNotNull() &
        col("store_num_dim").isNotNull()
    )
)

df_fact_valid = df_fact_checked.filter(col("fk_valid")).select("f.*")
df_fact_quarantine = df_fact_checked.filter(~col("fk_valid")).select("f.*")

print("Valid rows:", df_fact_valid.count())
print("Quarantined rows:", df_fact_quarantine.count())

StatementMeta(, 4db4e963-533c-4d89-8adf-fde8223bfd86, 8, Finished, Available, Finished, False)

Valid rows: 943
Quarantined rows: 0


In [7]:
#Adding an audit timestamp

df_fact_valid = df_fact_valid.withColumn("silver_loaded_at", current_timestamp())

StatementMeta(, 4db4e963-533c-4d89-8adf-fde8223bfd86, 9, Finished, Available, Finished, False)

In [8]:
#append-write valid rows and saving quarantine separately

# guard: only keep rows not already loaded (checked by trxn_id)
if spark.catalog.tableExists("silver_f_sls_t"):
    existing_ids = spark.table("silver_f_sls_t").select("trxn_id")
    df_fact_valid = df_fact_valid.join(existing_ids, "trxn_id", "left_anti")

print("New rows to append:", df_fact_valid.count())

df_fact_valid.write.format("delta").mode("append").saveAsTable("silver_f_sls_t")

if df_fact_quarantine.count() > 0:
    df_fact_quarantine = df_fact_quarantine.withColumn("silver_loaded_at", current_timestamp())
    df_fact_quarantine.write.format("delta").mode("append").saveAsTable("silver_f_sls_t_quarantine")

StatementMeta(, 4db4e963-533c-4d89-8adf-fde8223bfd86, 10, Finished, Available, Finished, False)

New rows to append: 943


In [9]:
spark.table("silver_f_sls_t").printSchema()

StatementMeta(, 4db4e963-533c-4d89-8adf-fde8223bfd86, 11, Finished, Available, Finished, False)

root
 |-- trxn_id: integer (nullable = true)
 |-- prod_num: string (nullable = true)
 |-- store_num: string (nullable = true)
 |-- cust_num: string (nullable = true)
 |-- trxn_dt: timestamp (nullable = true)
 |-- qty: integer (nullable = true)
 |-- amount: decimal(11,2) (nullable = true)
 |-- day: integer (nullable = true)
 |-- silver_loaded_at: timestamp (nullable = true)



In [1]:
spark.table("silver_f_sls_t").count()  

StatementMeta(, c858830d-428b-4ed4-bdaa-b01b4b1c81c5, 3, Finished, Available, Finished, False)

2132

In [2]:
%%sql
SELECT * FROM dbo.silver_f_sls_t
LIMIT 10;

StatementMeta(, a4abf5d5-2857-40ce-9bf2-d03cb64e50cd, 3, Finished, Available, Finished, False)

<Spark SQL result set with 10 rows and 9 fields>

In [1]:
%%sql
SELECT COUNT(*) FROM dbo.silver_f_sls_t;

StatementMeta(, a4abf5d5-2857-40ce-9bf2-d03cb64e50cd, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>